# Compare LLM weight compression formats on NPU with OpenVINO™ GenAI

Weight compression makes a large language model (LLM) small enough for a laptop, and the format to use depends on the device. The [OpenVINO GenAI guide for NPU](https://docs.openvino.ai/2026/openvino-workflow-generative/inference-with-genai/inference-with-genai-on-npu.html) asks for symmetric 4-bit weights, quantized channel-wise or in groups of 128.

Running on the device is one requirement. Answering like the original model is another.

This notebook shows a quick way to check the second one on the device you will use. It asks the original PyTorch model 20 short questions, asks each compressed format the same questions on the device you select, and counts the answers that start with the same line. Every run uses greedy decoding, so sampling does not add differences of its own.

In this notebook you will:

1. Generate reference answers with the original model in Hugging Face Transformers.
2. Prepare several weight formats of the model, exported with `optimum-cli` or downloaded from the Hugging Face Hub.
3. Run each format with the OpenVINO GenAI `LLMPipeline` on NPU, CPU or GPU.
4. Compare the answers with the reference answers.
5. Measure the generation speed of each format and try your own questions.

The default model is [Qwen3-0.6B](https://huggingface.co/Qwen/Qwen3-0.6B). [Qwen3-1.7B](https://huggingface.co/Qwen/Qwen3-1.7B) can be selected instead. Both are released under the Apache 2.0 license.

<img referrerpolicy="no-referrer-when-downgrade" src="https://static.scarf.sh/a.png?x-pxid=5b5a4db0-7875-4bfb-bdbd-01698b5b1a77&file=notebooks/llm-npu-weight-compression/llm-npu-weight-compression.ipynb" />


#### Table of contents:

- [Prerequisites](#Prerequisites)
- [Select the model, the device and the weight formats](#Select-the-model,-the-device-and-the-weight-formats)
- [Prepare the questions](#Prepare-the-questions)
- [Get reference answers from the original model](#Get-reference-answers-from-the-original-model)
- [Prepare the weight formats](#Prepare-the-weight-formats)
- [Run each format on the device](#Run-each-format-on-the-device)
- [Compare the answers](#Compare-the-answers)
- [Measure generation speed](#Measure-generation-speed)
- [Interactive demo](#Interactive-demo)
- [Cleanup](#Cleanup)

### Installation Instructions

This is a self-contained example that relies solely on its own code.

We recommend  running the notebook in a virtual environment. You only need a Jupyter server to start.
For details, please refer to [Installation Guide](https://github.com/openvinotoolkit/openvino_notebooks/blob/latest/README.md#-installation-guide).

## Prerequisites
[back to top ⬆️](#Table-of-contents:)

The first cell fetches the helper files of the repository and creates a virtual environment for this notebook. The second cell installs the packages into it.

To run on NPU, install the latest [NPU driver](https://docs.openvino.ai/2026/get-started/install-openvino/configurations/configurations-intel-npu.html) first. Without an NPU the notebook runs on CPU, where the same comparison works.

With the default model and formats, a first run on NPU downloads about 3.8 GB of model files, leaves about 6 GB on disk next to the notebook and uses about 9 GB of RAM at its peak. With Qwen3-1.7B and the default formats, the model files take about 11 GB on disk and a first run on NPU uses about 20 GB of RAM at its peak. Every selected format stays loaded until the end of the notebook.

In [ ]:
import urllib.request
from pathlib import Path

# Helpers come from `utils` of the cloned repository, otherwise from GitHub; local copies are overwritten
helper_files = ["notebook_utils.py", "cmd_helper.py"]
repo_utils = Path("../../utils")
in_repo = (repo_utils / "notebook_utils.py").is_file()
if in_repo:
    helper_contents = {helper_file: (repo_utils / helper_file).read_bytes() for helper_file in helper_files}
else:
    try:
        helper_contents = {}
        for helper_file in helper_files:
            url = f"https://raw.githubusercontent.com/openvinotoolkit/openvino_notebooks/latest/utils/{helper_file}"
            with urllib.request.urlopen(url, timeout=10) as response:
                helper_contents[helper_file] = response.read()
    except OSError as error:
        if any(not Path(helper_file).is_file() for helper_file in helper_files):
            raise
        print(f"Could not update helper files, using the existing copies: {error}")
        helper_contents = {}

for helper_file, content in helper_contents.items():
    Path(helper_file).write_bytes(content)

from notebook_utils import setup_notebook_venv

# Requirements installed below go to a notebook-specific virtual environment in the `.venv` folder
setup_notebook_venv()

# Read more about telemetry collection at https://github.com/openvinotoolkit/openvino_notebooks?tab=readme-ov-file#-telemetry
from notebook_utils import collect_telemetry

collect_telemetry("llm-npu-weight-compression.ipynb")

In [ ]:
%pip install -q "openvino>=2026.4.1" "openvino-genai>=2026.4.1" "openvino-tokenizers>=2026.4.1"
%pip install -q --extra-index-url https://download.pytorch.org/whl/cpu "optimum-intel>=2.2.0" "nncf>=3.4.0" "torch>=2.14.0" "transformers>=5.5.4" "huggingface_hub>=1.21.0" "datasets>=4.8.5"
%pip install -q -U gradio

## Select the model, the device and the weight formats
[back to top ⬆️](#Table-of-contents:)

Select the model, the device to check and the formats to compare. NPU is the default device when OpenVINO finds one, otherwise CPU.

The formats are described in the section [Prepare the weight formats](#Prepare-the-weight-formats). In a first run with the default model and formats on a laptop with an Intel® Core™ Ultra 9 386H processor, the package installation took about seven minutes, the INT4-CW export about one minute, and loading and running the four formats on NPU a little over two minutes. You can add or remove formats here.

With **Use preconverted models** selected, a format is downloaded from the Hugging Face Hub when a converted model exists for it, and exported on your machine otherwise. Clear the checkbox to export every format yourself.

In [ ]:
import ipywidgets as widgets

# The original models on the Hugging Face Hub and the revision of each that this notebook downloads
ORIGINAL_MODELS = {
    "Qwen/Qwen3-0.6B": "c1899de289a04d12100db370d81485cdf75e47ca",
    "Qwen/Qwen3-1.7B": "70d244cc86ccca08cf5af4e1e306ecf908b1ad5e",
}

model_id = widgets.Dropdown(options=list(ORIGINAL_MODELS), value="Qwen/Qwen3-0.6B", description="Model:")
model_id

In [ ]:
import openvino as ov
from notebook_utils import device_widget

default_device = "NPU" if "NPU" in ov.Core().available_devices else "CPU"
device = device_widget(default=default_device, exclude=["AUTO"])
device

In [ ]:
from IPython.display import display

# The weight formats this notebook can compare. For each format:
# - "export_args": the options of `optimum-cli export openvino`, as `additional_args` of `cmd_helper.optimum_cli`
#   (an option without a value maps to None)
# - "preconverted": for each original model, a converted model on the Hugging Face Hub as (repo id, revision), if there is one
WEIGHT_FORMATS = {
    "INT4-GQ": {
        "export_args": {"task": "text-generation-with-past", "weight-format": "int4", "sym": None, "ratio": 1.0, "group-size": 128},
        "preconverted": {
            "Qwen/Qwen3-0.6B": ("mlboydaisuke/Qwen3-0.6B-int4-sym-g128-ov", "71a1efa5724032f75f4694b97556770f772b8b15"),
            "Qwen/Qwen3-1.7B": ("mlboydaisuke/Qwen3-1.7B-int4-sym-g128-ov", "fd492dc43e21b7a4378302fa7c88f87d9173a2ba"),
        },
    },
    "INT4-CW": {
        "export_args": {"task": "text-generation-with-past", "weight-format": "int4", "sym": None, "ratio": 1.0, "group-size": -1},
        "preconverted": {},
    },
    "NF4-CW": {
        "export_args": {"task": "text-generation-with-past", "weight-format": "nf4", "sym": None, "ratio": 1.0, "group-size": -1},
        "preconverted": {},
    },
    "INT4-GQ, data-aware": {
        "export_args": {
            "task": "text-generation-with-past",
            "weight-format": "int4",
            "sym": None,
            "ratio": 1.0,
            "group-size": 128,
            "awq": None,
            "scale-estimation": None,
            "dataset": "wikitext2",
        },
        "preconverted": {},
    },
    "INT4-CW, data-aware": {
        "export_args": {
            "task": "text-generation-with-past",
            "weight-format": "int4",
            "sym": None,
            "ratio": 1.0,
            "group-size": -1,
            "awq": None,
            "scale-estimation": None,
            "dataset": "wikitext2",
        },
        "preconverted": {},
    },
    "INT8": {
        "export_args": {"task": "text-generation-with-past", "weight-format": "int8", "sym": None},
        "preconverted": {
            "Qwen/Qwen3-0.6B": ("mlboydaisuke/Qwen3-0.6B-int8-sym-ov", "54803fd2b567b03471f1f2892af1d558b6f09b83"),
            "Qwen/Qwen3-1.7B": ("mlboydaisuke/Qwen3-1.7B-int8-sym-ov", "a269b0efd7e80b94a2a9804eda55d3679a3adf3f"),
        },
    },
    "FP16": {
        "export_args": {"task": "text-generation-with-past", "weight-format": "fp16"},
        "preconverted": {
            "Qwen/Qwen3-0.6B": ("OpenVINO/Qwen3-0.6B-fp16-ov", "36e37311b8e5763bfedb7da3e35925c67417c33e"),
            "Qwen/Qwen3-1.7B": ("OpenVINO/Qwen3-1.7B-fp16-ov", "4040dedc9515167d1689346577534968c07570f9"),
        },
    },
}

# The formats selected by default
DEFAULT_FORMATS = ("INT4-GQ", "INT4-CW", "INT8", "FP16")

formats = widgets.SelectMultiple(
    options=list(WEIGHT_FORMATS),
    value=DEFAULT_FORMATS,
    rows=len(WEIGHT_FORMATS),
    description="Formats:",
)
use_preconverted = widgets.Checkbox(value=True, description="Use preconverted models")
display(widgets.VBox([formats, use_preconverted]))

## Prepare the questions
[back to top ⬆️](#Table-of-contents:)

The check uses 20 short questions that ask for a short answer, for example "What is the capital of France? Answer with one word." Short answers keep the comparison simple. Two answers agree when their first lines are the same. Before the comparison, each first line loses the spaces, quotes, `` ` `` and `*` characters around it and a final `.`, `!` or `?`. This is repeated until the line stops changing, and each run of spaces inside the line becomes one space.

The check does not ask whether an answer is correct. It asks whether a compressed model answers like the original one, so an answer of the original model is the reference even when it is wrong.

Each question is sent as one user turn in the chat template of the model. Qwen3 can think before it answers. The template is rendered with `enable_thinking=False`, so the answer starts with the first generated token. Each answer is limited to 48 new tokens.

Greedy decoding is used here to make the runs comparable. For normal use, follow the sampling settings recommended in the model card.

The questions come from [windows-model-zoo](https://github.com/john-rocky/windows-model-zoo), which uses them to test converted models on a Windows laptop.

In [ ]:
QUESTIONS = (
    "What is the capital of France? Answer with one word.",
    "What is 7 + 5? Reply with only the number.",
    "What is 12 * 3? Reply with only the number.",
    'Spell the word "cat" backwards. Reply with only the result.',
    "Which planet is known as the Red Planet? Answer with one word.",
    "How many days are in a week? Reply with only the number.",
    'What is the opposite of "hot"? Answer with one word.',
    'Translate "thank you" into Japanese. Reply with only the translation.',
    "What color do you get by mixing blue and yellow? Answer with one word.",
    "What is the chemical symbol for gold? Reply with only the symbol.",
    "Is 17 a prime number? Answer yes or no.",
    "What is the largest ocean on Earth? Reply with only the name.",
    'Who wrote "Romeo and Juliet"? Reply with only the name.',
    "What is 100 divided by 4? Reply with only the number.",
    "What comes next: 2, 4, 6, 8? Reply with only the number.",
    "At what temperature in Celsius does water freeze? Reply with only the number.",
    'Which is heavier, a kilogram of iron or a kilogram of feathers? Answer "iron", "feathers" or "same".',
    "What is the square root of 81? Reply with only the number.",
    'Write the word "hello" in uppercase. Reply with only the word.',
    "How many legs does a spider have? Reply with only the number.",
)
MAX_NEW_TOKENS = 48

questions = QUESTIONS


def first_line(text: str) -> str:
    """
    Return the first line of an answer in the form in which answers are compared: without the white space around it,
    without the quotes, backticks and asterisks around it and without a final ".", "!" or "?" (removed again until
    nothing changes), and with each run of white space inside it replaced by one space. Upper and lower case are kept,
    so "hello" and "HELLO" are different answers.

    :param text: An answer.
    :return: The first line of the answer in that form, an empty string for an empty answer.
    """
    lines = (text or "").strip().splitlines()
    line = lines[0] if lines else ""
    previous = None
    while line != previous:
        previous = line
        line = line.strip().strip("\"'`*").rstrip(".!?")
    return " ".join(line.split())

## Get reference answers from the original model
[back to top ⬆️](#Table-of-contents:)

The reference is the original checkpoint, loaded with Hugging Face Transformers in FP32 on CPU. The cell below downloads the checkpoint, renders each question with the chat template and generates the answer. It keeps the rendered prompts, because every compressed format will get exactly these prompts.

In [7]:
import gc

import torch
from huggingface_hub import snapshot_download
from transformers import AutoModelForCausalLM, AutoTokenizer, PreTrainedModel, PreTrainedTokenizerBase

MODEL_DIR = Path("model")
# Files of the original checkpoint that the reference and the export need (not the README or the license text)
ORIGINAL_FILES = ["*.json", "*.safetensors", "merges.txt"]


def render_prompt(tokenizer: PreTrainedTokenizerBase, question: str) -> str:
    """
    Render a question as one user turn in the chat template of the model, with thinking switched off.

    :param tokenizer: Tokenizer of the original model.
    :param question: The question.
    :return: The prompt as text, up to the start of the answer.
    """
    return tokenizer.apply_chat_template([{"role": "user", "content": question}], tokenize=False, add_generation_prompt=True, enable_thinking=False)


def generate_reference(model: PreTrainedModel, tokenizer: PreTrainedTokenizerBase, question: str, max_new_tokens: int) -> tuple[str, str]:
    """
    Answer a question with the original model and greedy decoding.

    :param model: The original model.
    :param tokenizer: Tokenizer of the original model.
    :param question: The question.
    :param max_new_tokens: The most tokens to generate.
    :return: The rendered prompt and the answer without special tokens.
    """
    prompt = render_prompt(tokenizer=tokenizer, question=question)
    eos = model.generation_config.eos_token_id
    eos_ids = [int(token) for token in (eos if isinstance(eos, (list, tuple)) else [eos]) if token is not None]
    pad_id = tokenizer.pad_token_id if tokenizer.pad_token_id is not None else eos_ids[0]
    input_ids = tokenizer(prompt, add_special_tokens=False, return_tensors="pt").input_ids
    with torch.inference_mode():
        # Plain greedy decoding: these arguments replace the values of the checkpoint's generation config, and its
        # sampling settings are unset, as greedy decoding does not use them
        output = model.generate(
            input_ids,
            attention_mask=torch.ones_like(input_ids),
            do_sample=False,
            num_beams=1,
            repetition_penalty=1.0,
            temperature=None,
            top_p=None,
            top_k=None,
            max_new_tokens=max_new_tokens,
            eos_token_id=eos_ids,
            pad_token_id=pad_id,
        )
    return prompt, tokenizer.decode(output[0, input_ids.shape[1] :].tolist(), skip_special_tokens=True)


# The model selected above. The following cells use this value: after selecting another model, run again from this cell
original_id = model_id.value
model_name = original_id.split("/")[-1]
original_dir = MODEL_DIR / model_name / "original"
snapshot_download(repo_id=original_id, revision=ORIGINAL_MODELS[original_id], local_dir=original_dir, allow_patterns=ORIGINAL_FILES)

tokenizer = AutoTokenizer.from_pretrained(original_dir)
reference_model = AutoModelForCausalLM.from_pretrained(original_dir, dtype=torch.float32).eval()

prompts = []
reference_answers = []
for question in questions:
    prompt, answer = generate_reference(model=reference_model, tokenizer=tokenizer, question=question, max_new_tokens=MAX_NEW_TOKENS)
    prompts.append(prompt)
    reference_answers.append(answer)
# The compressed formats do not need the original model: release its memory
del reference_model
gc.collect()

print(f"{len(questions)} reference answers of {original_id}")
for question, answer in zip(questions, reference_answers):
    print(f"{question} -> {answer!r}")

20 reference answers of Qwen/Qwen3-0.6B
What is the capital of France? Answer with one word. -> 'Paris'
What is 7 + 5? Reply with only the number. -> '7 + 5 = 12'
What is 12 * 3? Reply with only the number. -> '12 * 3 = 36'
Spell the word "cat" backwards. Reply with only the result. -> 'cat'
Which planet is known as the Red Planet? Answer with one word. -> 'Mars'
How many days are in a week? Reply with only the number. -> '1'
What is the opposite of "hot"? Answer with one word. -> 'hot'
Translate "thank you" into Japanese. Reply with only the translation. -> 'ありがとう'
What color do you get by mixing blue and yellow? Answer with one word. -> 'blue'
What is the chemical symbol for gold? Reply with only the symbol. -> 'Au'
Is 17 a prime number? Answer yes or no. -> 'Yes, 17 is a prime number.'
What is the largest ocean on Earth? Reply with only the name. -> 'Pacific Ocean'
Who wrote "Romeo and Juliet"? Reply with only the name. -> 'Romeo and Juliet'
What is 100 divided by 4? Reply with only

## Prepare the weight formats
[back to top ⬆️](#Table-of-contents:)

The table lists the formats this notebook can compare and the options that `optimum-cli export openvino` gets for each of them. Every export also gets `--task text-generation-with-past`.

| Format | Export options | What it is |
|---|---|---|
| INT4-GQ | `--weight-format int4 --sym --ratio 1.0 --group-size 128` | 4-bit weights with group quantization: each group of 128 weights shares its quantization parameters |
| INT4-CW | `--weight-format int4 --sym --ratio 1.0 --group-size -1` | 4-bit weights with channel-wise quantization |
| NF4-CW | `--weight-format nf4 --sym --ratio 1.0 --group-size -1` | 4-bit weights in the NF4 data type, for the NPU of Intel® Core™ Ultra Processors Series 2 and newer |
| INT4-GQ, data-aware | the INT4-GQ options and `--awq --scale-estimation --dataset wikitext2` | INT4-GQ with AWQ and scale estimation, which use a dataset to reduce the loss from compression and take longer to export |
| INT4-CW, data-aware | the INT4-CW options and `--awq --scale-estimation --dataset wikitext2` | INT4-CW with the same two methods |
| INT8 | `--weight-format int8 --sym` | 8-bit weights |
| FP16 | `--weight-format fp16` | 16-bit floating point weights, no further compression |

The five 4-bit formats follow the settings that the [NPU guide](https://docs.openvino.ai/2026/openvino-workflow-generative/inference-with-genai/inference-with-genai-on-npu.html) requires for LLMs. The guide recommends group quantization for smaller models. About channel-wise quantization it says that it generally gives the best performance but may reduce accuracy, and that data-aware compression can compensate for the loss.

INT8 and FP16 are outside that list. They are larger, and they are here as reference points: they show how the device answers when the weights are compressed less. INT8 is exported with `--sym` because the guide asks for symmetric weights. Without this option, 8-bit weights are compressed asymmetrically.

The export is done by [Optimum Intel](https://huggingface.co/docs/optimum/intel/index), which compresses the weights with [NNCF](https://github.com/openvinotoolkit/nncf). The [weight compression guide](https://docs.openvino.ai/2026/openvino-workflow/model-optimization-guide/weight-compression.html) explains the options. It also notes that low-bit compression may reduce accuracy more for models with less than 1B parameters, such as the default model of this notebook, than for larger models.

Converted models are on the Hugging Face Hub for three formats: INT4-GQ ([Qwen3-0.6B](https://huggingface.co/mlboydaisuke/Qwen3-0.6B-int4-sym-g128-ov), [Qwen3-1.7B](https://huggingface.co/mlboydaisuke/Qwen3-1.7B-int4-sym-g128-ov)), INT8 ([Qwen3-0.6B](https://huggingface.co/mlboydaisuke/Qwen3-0.6B-int8-sym-ov), [Qwen3-1.7B](https://huggingface.co/mlboydaisuke/Qwen3-1.7B-int8-sym-ov)) and FP16 ([Qwen3-0.6B](https://huggingface.co/OpenVINO/Qwen3-0.6B-fp16-ov), [Qwen3-1.7B](https://huggingface.co/OpenVINO/Qwen3-1.7B-fp16-ov)). The other formats are always exported on your machine.

In [8]:
import os
import re
import subprocess

from cmd_helper import optimum_cli

# Set to True to stop at the first weight format that cannot be prepared or run, instead of showing it in the result table
STOP_ON_ERROR = False

# Files of a converted model on the Hugging Face Hub that LLMPipeline needs (not the README or other files of the repository)
PRECONVERTED_FILES = [
    "openvino_*",
    "config.json",
    "generation_config.json",
    "tokenizer.json",
    "tokenizer_config.json",
    "special_tokens_map.json",
    "added_tokens.json",
    "vocab.json",
    "merges.txt",
    "chat_template.jinja",
]
# The files of an OpenVINO IR folder that LLMPipeline loads: a folder that has all of them is used as it is
IR_FILES = [
    "openvino_model.xml",
    "openvino_model.bin",
    "openvino_tokenizer.xml",
    "openvino_tokenizer.bin",
    "openvino_detokenizer.xml",
    "openvino_detokenizer.bin",
]


def weight_format_dir(model_dir: Path, format_name: str, from_hub: bool) -> Path:
    """
    Give the folder of one weight format. A converted model downloaded from the Hugging Face Hub and a model exported on
    this machine have folders of their own, so one is never used in place of the other: for example, "INT4-GQ, data-aware"
    is exported to "int4-gq-data-aware", and the INT4-GQ model from the Hub goes to "int4-gq-hub".

    :param model_dir: Folder for the formats of the original model, for example `model/Qwen3-0.6B`.
    :param format_name: Name of the weight format.
    :param from_hub: Whether the folder is for the converted model from the Hub.
    :return: The folder.
    """
    name = re.sub(r"[^a-z0-9]+", "-", format_name.lower()).strip("-")
    return model_dir / (f"{name}-hub" if from_hub else name)


def error_line(error: Exception) -> str:
    """
    Say in one line why a step failed: for a failed `optimum-cli` run its exit code and the last line of its error
    output, otherwise the type of the error and the first line of its message that is not a source location.

    :param error: The error.
    :return: One line of text.
    """
    if isinstance(error, subprocess.CalledProcessError):
        output = error.stderr.decode(errors="replace") if isinstance(error.stderr, bytes) else str(error.stderr or "")
        lines = [line.strip() for line in output.splitlines() if line.strip()]
        return f"optimum-cli exited with code {error.returncode}" + (f": {lines[-1]}" if lines else "")
    lines = [line.strip() for line in str(error).splitlines() if line.strip() and not line.strip().startswith("Exception from ")]
    return f"{type(error).__name__}: {lines[0]}" if lines else type(error).__name__


def prepare_weight_format(weight_format: dict, original_id: str, original_dir: Path, folder: Path, from_hub: bool) -> str:
    """
    Make the OpenVINO IR of one weight format available in its folder: use the folder when it already holds a complete
    IR, otherwise download the converted model from the Hugging Face Hub or export the format from the original checkpoint.

    :param weight_format: The entry of the format in `WEIGHT_FORMATS`.
    :param original_id: Hugging Face id of the original model.
    :param original_dir: Folder with the original checkpoint.
    :param folder: Folder of the format, from `weight_format_dir`.
    :param from_hub: Whether to download the converted model from the Hub instead of exporting the format.
    :return: How the IR was made available: "found", "downloaded" or "exported".
    """
    if all((folder / name).exists() for name in IR_FILES):
        return "found"
    if from_hub:
        repo_id, revision = weight_format["preconverted"][original_id]
        snapshot_download(repo_id=repo_id, revision=revision, local_dir=folder, allow_patterns=PRECONVERTED_FILES)
        return "downloaded"
    # optimum-cli runs in a child process whose output goes to a pipe. On Windows with a code page that is not UTF-8 (for
    # example cp932 on a Japanese system), the progress bar of the weight compression cannot write some of its characters
    # there and the export stops with a UnicodeEncodeError, so the child process writes UTF-8
    os.environ.setdefault("PYTHONIOENCODING", "utf-8")
    # The paths go with "/", so the command that the cell shows is the same on Windows
    optimum_cli(model_id=original_dir.as_posix(), output_dir=folder.as_posix(), additional_args=weight_format["export_args"])
    return "exported"


selected_formats = [name for name in WEIGHT_FORMATS if name in formats.value]
if not selected_formats:
    raise ValueError("No weight format is selected. Select at least one format above and run the notebook from this cell again.")

format_dirs = {}
weights_mb = {}
not_prepared = {}
for format_name in selected_formats:
    # The converted model from the Hub is used only when the checkbox is set and the Hub has one for the selected model
    from_hub = use_preconverted.value and original_id in WEIGHT_FORMATS[format_name]["preconverted"]
    format_dir = weight_format_dir(model_dir=MODEL_DIR / model_name, format_name=format_name, from_hub=from_hub)
    try:
        how = prepare_weight_format(
            weight_format=WEIGHT_FORMATS[format_name], original_id=original_id, original_dir=original_dir, folder=format_dir, from_hub=from_hub
        )
    except Exception as error:  # a format that cannot be prepared appears in the result table, the other formats go on
        not_prepared[format_name] = f"could not be prepared: {error_line(error)}"
        if STOP_ON_ERROR:
            raise RuntimeError(f"{format_name}: {not_prepared[format_name]}") from error
        print(f"{format_name}: {not_prepared[format_name]}")
        continue
    format_dirs[format_name] = format_dir
    weights_mb[format_name] = (format_dir / "openvino_model.bin").stat().st_size / 1e6
    print(f"{format_name}: {how}, weights {weights_mb[format_name]:.1f} MB, in {format_dir.as_posix()}")

INT4-GQ: downloaded, weights 383.4 MB, in model/Qwen3-0.6B/int4-gq-hub


**Export command:**

`optimum-cli export openvino --model model/Qwen3-0.6B/original model/Qwen3-0.6B/int4-cw --task text-generation-with-past --weight-format int4 --sym --ratio 1.0 --group-size -1`

INT4-CW: exported, weights 377.2 MB, in model/Qwen3-0.6B/int4-cw


INT8: downloaded, weights 597.2 MB, in model/Qwen3-0.6B/int8-hub


FP16: downloaded, weights 1192.1 MB, in model/Qwen3-0.6B/fp16-hub


## Run each format on the device
[back to top ⬆️](#Table-of-contents:)

Each format is loaded with `LLMPipeline` on the selected device and answers the rendered prompts from the reference step. The prompts are tokenized without adding special tokens and the chat template of the pipeline is switched off, so that every format gets the same token ids as the original model. The notebook checks this for each format before it counts the answers.

The pipelines use the default properties of the device, so the result shows what you get without tuning. That includes the inference precision: the device selects it, and the weight format does not ([Precision Control](https://docs.openvino.ai/2026/openvino-workflow/running-inference/optimize-inference/precision-control.html)). One property is added, `CACHE_DIR`. Compiling an LLM for NPU can take a while, so the compiled models are stored in the `cache` folder and the next start is faster.

A format that the device cannot load or run does not stop the notebook. It appears in the result table with its error. So does a format whose tokenizer turns a prompt into other token ids. If loading a format ends the Python process instead of raising an error, the kernel dies. Run the cells up to the format selection again, remove that format from the selection, and run all the cells after it.

In [9]:
from importlib.metadata import version

# The NPU property with the driver version, printed when the NPU plugin can report it
NPU_DRIVER_PROPERTY = "NPU_DRIVER_VERSION"
# The property with the precision in which the device runs the model, printed when the device has it
PRECISION_PROPERTY = "INFERENCE_PRECISION_HINT"

for package in ("openvino", "openvino-genai", "optimum-intel", "nncf", "transformers", "torch"):
    print(f"{package}: {version(package)}")
core = ov.Core()
print(f"{device.value}: {core.get_property(device.value, 'FULL_DEVICE_NAME')}")
if PRECISION_PROPERTY in core.get_property(device.value, "SUPPORTED_PROPERTIES"):
    print(f"{PRECISION_PROPERTY}: {core.get_property(device.value, PRECISION_PROPERTY)}")
if device.value == "NPU":
    try:
        print(f"{NPU_DRIVER_PROPERTY}: {core.get_property('NPU', NPU_DRIVER_PROPERTY)}")
    except Exception:
        pass  # the driver version is shown only when the plugin can report it

openvino: 2026.4.1
openvino-genai: 2026.4.1.0
optimum-intel: 2.2.0
nncf: 3.4.0
transformers: 5.5.4
torch: 2.15.0+cpu
NPU: Intel(R) AI Boost
INFERENCE_PRECISION_HINT: <Type: 'float16'>
NPU_DRIVER_VERSION: 1005540


In [10]:
import openvino_genai as ov_genai

# The device of this run: the cells below name this device, also when another device is selected above later
run_device = device.value
CACHE_DIR = Path("cache")


def load_pipeline(model_dir: Path, device: str, cache_dir: Path) -> ov_genai.LLMPipeline:
    """
    Load an OpenVINO IR folder with LLMPipeline on a device, with the compiled model stored in a cache folder.

    :param model_dir: The OpenVINO IR folder.
    :param device: The device, for example "NPU".
    :param cache_dir: Folder for the compiled model (`CACHE_DIR`).
    :return: The pipeline.
    """
    return ov_genai.LLMPipeline(str(model_dir), device, CACHE_DIR=str(cache_dir))


def prompt_token_ids(pipe_tokenizer: ov_genai.Tokenizer, prompt: str) -> list[int]:
    """
    Tokenize a rendered prompt with the tokenizer of a pipeline as `generate_answer` does, without special tokens.

    :param pipe_tokenizer: Tokenizer of the pipeline.
    :param prompt: Prompt rendered by `render_prompt`.
    :return: The token ids.
    """
    # `input_ids` of the result is an openvino.Tensor of shape [1, number of tokens]; its `data` is a NumPy array
    return pipe_tokenizer.encode(prompt, add_special_tokens=False).input_ids.data.flatten().tolist()


def count_token_differences(pipe: ov_genai.LLMPipeline, prompts: list[str], reference_ids: list[list[int]]) -> int:
    """
    Count the prompts that the tokenizer of a pipeline turns into other token ids than the tokenizer of the original model.

    :param pipe: The pipeline.
    :param prompts: Prompts rendered by `render_prompt`.
    :param reference_ids: Token ids of each prompt from the tokenizer of the original model.
    :return: The number of prompts with other token ids.
    """
    pipe_tokenizer = pipe.get_tokenizer()
    return sum(prompt_token_ids(pipe_tokenizer=pipe_tokenizer, prompt=prompt) != ids for prompt, ids in zip(prompts, reference_ids))


def generate_answer(pipe: ov_genai.LLMPipeline, prompt: str, max_new_tokens: int) -> tuple[str, ov_genai.PerfMetrics]:
    """
    Answer a rendered prompt with greedy decoding. The prompt is tokenized without special tokens and the chat template
    of the pipeline is not applied, so the pipeline gets the same token ids as the original model.

    :param pipe: The pipeline.
    :param prompt: Prompt rendered by `render_prompt`.
    :param max_new_tokens: The most tokens to generate.
    :return: The answer without special tokens and the performance metrics of the generation.
    """
    config = pipe.get_generation_config()
    config.max_new_tokens = max_new_tokens
    config.do_sample = False
    config.num_beams = 1
    config.repetition_penalty = 1.0
    config.apply_chat_template = False
    pipe_tokenizer = pipe.get_tokenizer()
    result = pipe.generate(pipe_tokenizer.encode(prompt, add_special_tokens=False), config)
    return pipe_tokenizer.decode(list(result.tokens[0]), skip_special_tokens=True), result.perf_metrics


# Token ids of each prompt from the tokenizer of the original model: a format is compared only when it gets the same ids
reference_prompt_ids = [tokenizer(prompt, add_special_tokens=False).input_ids for prompt in prompts]

# Release the pipelines of an earlier run of this cell before loading them again
if "pipelines" in globals():
    pipelines.clear()
    gc.collect()
pipelines = {}
answers = {}
not_run = {}
for format_name in format_dirs:
    try:
        pipelines[format_name] = load_pipeline(
            model_dir=format_dirs[format_name], device=run_device, cache_dir=CACHE_DIR / model_name / format_dirs[format_name].name / run_device
        )
        differing = count_token_differences(pipe=pipelines[format_name], prompts=prompts, reference_ids=reference_prompt_ids)
        if differing:
            pipelines.pop(format_name)
            not_run[format_name] = f"prompt tokens differ from the original model's for {differing} of {len(prompts)} prompts"
        else:
            answers[format_name] = [generate_answer(pipe=pipelines[format_name], prompt=prompt, max_new_tokens=MAX_NEW_TOKENS)[0] for prompt in prompts]
    except Exception as error:  # a format that the device cannot load or run appears in the result table, the other formats go on
        pipelines.pop(format_name, None)
        not_run[format_name] = f"could not run on {run_device}: {error_line(error)}"
        if STOP_ON_ERROR:
            raise RuntimeError(f"{format_name}: {not_run[format_name]}") from error
    if format_name in not_run:
        if STOP_ON_ERROR:
            raise RuntimeError(f"{format_name}: {not_run[format_name]}")
        print(f"{format_name}: {not_run[format_name]}")
        continue
    print(f"{format_name}: loaded on {run_device}, {len(prompts)} answers")

INT4-GQ: loaded on NPU, 20 answers


INT4-CW: loaded on NPU, 20 answers


INT8: loaded on NPU, 20 answers


FP16: loaded on NPU, 20 answers


## Compare the answers
[back to top ⬆️](#Table-of-contents:)

The table shows the size of the weights of each format and the number of answers that agree with the original model. Below the table, every answer that differs is printed next to the reference answer.

How to read the result:

- A format with all or nearly all answers the same keeps the behavior of the original model in this check. Among such formats, choose by size and speed.
- A format with many different answers changes what the model says on this device. Read the answers that differ before you decide whether the format is good enough for your use.
- The result is for the selected device. To see the same formats on another device, select it above and run the cells again from [Run each format on the device](#Run-each-format-on-the-device).
- 20 questions are a quick check, not a benchmark. For a broader test, [WhoWhatBench](https://github.com/openvinotoolkit/openvino.genai/tree/master/tools/who_what_benchmark) computes a similarity score between the outputs of a baseline model and an optimized model on a dataset. Before you ship a model, repeat the comparison with prompts from your own application.

In [11]:
from IPython.display import Markdown


def table_text(text: str) -> str:
    """
    Make a text safe for one cell of a Markdown table.

    :param text: The text.
    :return: The text on one line, with "|" escaped.
    """
    return " ".join(text.split()).replace("|", "\\|")


rows = ["| Format | Weights (MB) | Answers the same as the original model |", "|---|---|---|"]
for format_name in selected_formats:
    size = f"{weights_mb[format_name]:.1f}" if format_name in weights_mb else "-"
    if format_name in answers:
        same = sum(first_line(answer) == first_line(reference) for answer, reference in zip(answers[format_name], reference_answers))
        result = f"{same} / {len(questions)}"
    else:
        result = table_text(not_prepared.get(format_name) or not_run.get(format_name, "not run"))
    rows.append(f"| {format_name} | {size} | {result} |")
display(Markdown("\n".join(rows)))

for format_name in answers:
    differing = [
        (question, reference, answer)
        for question, reference, answer in zip(questions, reference_answers, answers[format_name])
        if first_line(answer) != first_line(reference)
    ]
    print(f"\n{format_name} on {run_device}: {len(differing)} of {len(questions)} answers differ from the original model")
    for question, reference, answer in differing:
        print(f"  Question:       {question}")
        print(f"  Original model: {reference!r}")
        print(f"  {format_name + ':':<15} {answer!r}")

| Format | Weights (MB) | Answers the same as the original model |
|---|---|---|
| INT4-GQ | 383.4 | 9 / 20 |
| INT4-CW | 377.2 | 3 / 20 |
| INT8 | 597.2 | 18 / 20 |
| FP16 | 1192.1 | 20 / 20 |


INT4-GQ on NPU: 11 of 20 answers differ from the original model
  Question:       What is 12 * 3? Reply with only the number.
  Original model: '12 * 3 = 36'
  INT4-GQ:        '12 * 3 = 36. So, the answer is 36.'
  Question:       How many days are in a week? Reply with only the number.
  Original model: '1'
  INT4-GQ:        '10.'
  Question:       Translate "thank you" into Japanese. Reply with only the translation.
  Original model: 'ありがとう'
  INT4-GQ:        'ありがとう。'
  Question:       What color do you get by mixing blue and yellow? Answer with one word.
  Original model: 'blue'
  INT4-GQ:        'The answer is: **blue**.'
  Question:       What is the largest ocean on Earth? Reply with only the name.
  Original model: 'Pacific Ocean'
  INT4-GQ:        'Terra.'
  Question:       What is 100 divided by 4? Reply with only the number.
  Original model: '100 divided by 4 is 25.'
  INT4-GQ:        '100 divided by 4 is 25. So, the answer is 25.'
  Question:       What comes next: 2, 4, 6

## Measure generation speed
[back to top ⬆️](#Table-of-contents:)

The answers above are too short to measure speed. The cell below lets each format generate a longer text and reads the time to the first token and the generation throughput from the performance metrics of the pipeline. The first generation of each format is a warm-up and is not counted.

On NPU the numbers reflect the default settings of the pipeline. A prompt shorter than `MAX_PROMPT_LEN` (1024 tokens by default) takes the same time to the first token as a prompt of the full length. The default `GENERATE_HINT`, `FAST_COMPILE`, compiles the model faster at the expense of generation speed. The [NPU guide](https://docs.openvino.ai/2026/openvino-workflow-generative/inference-with-genai/inference-with-genai-on-npu.html) describes these options.

Other programs on the machine change these numbers. Close heavy applications and run the cell more than once before you compare formats.

In [ ]:
SPEED_PROMPT = "Explain in a few paragraphs how a refrigerator keeps food cold."
SPEED_MAX_NEW_TOKENS = 128


def measure_speed(pipe: ov_genai.LLMPipeline, prompt: str, max_new_tokens: int) -> tuple[float, float, int]:
    """
    Measure the time to the first token and the generation throughput of a pipeline. The first generation is a warm-up
    and is not counted; the numbers come from the second one.

    :param pipe: The pipeline.
    :param prompt: Prompt rendered by `render_prompt`.
    :param max_new_tokens: The most tokens to generate.
    :return: Time to the first token in ms, throughput in tokens per second and the number of generated tokens.
    """
    generate_answer(pipe=pipe, prompt=prompt, max_new_tokens=max_new_tokens)
    _, metrics = generate_answer(pipe=pipe, prompt=prompt, max_new_tokens=max_new_tokens)
    return metrics.get_ttft().mean, metrics.get_throughput().mean, metrics.get_num_generated_tokens()


speed_prompt = render_prompt(tokenizer=tokenizer, question=SPEED_PROMPT)
rows = ["| Format | Time to first token (ms) | Throughput (tokens/s) | Generated tokens |", "|---|---|---|---|"]
for format_name in pipelines:
    try:
        ttft_ms, tokens_per_second, generated = measure_speed(pipe=pipelines[format_name], prompt=speed_prompt, max_new_tokens=SPEED_MAX_NEW_TOKENS)
    except Exception as error:  # a failed measurement appears in the table, the other formats go on
        if STOP_ON_ERROR:
            raise RuntimeError(f"{format_name}: the speed could not be measured on {run_device}: {error_line(error)}") from error
        rows.append(f"| {format_name} | {table_text(error_line(error))} | - | - |")
        continue
    rows.append(f"| {format_name} | {ttft_ms:.0f} | {tokens_per_second:.1f} | {generated} |")
print(f"Device: {run_device}, prompt: {SPEED_PROMPT!r}, at most {SPEED_MAX_NEW_TOKENS} new tokens")
display(Markdown("\n".join(rows)))

## Interactive demo
[back to top ⬆️](#Table-of-contents:)

Type your own question and see the answers of all loaded formats side by side. The demo uses the same settings as the comparison: one user turn, thinking switched off, greedy decoding and at most 48 new tokens, so a longer answer is cut off. On NPU the pipeline accepts prompts of up to 1024 tokens by default.

In [ ]:
import gradio as gr


def make_demo(pipelines: dict, tokenizer: PreTrainedTokenizerBase, max_new_tokens: int) -> gr.Interface:
    """
    Build a Gradio demo that sends one question to all loaded formats and shows their answers next to each other.

    :param pipelines: The loaded pipelines by format name.
    :param tokenizer: Tokenizer of the original model, which renders the question in the chat template.
    :param max_new_tokens: The most tokens to generate for each answer.
    :return: The demo.
    """
    format_names = list(pipelines)

    def answer_all(question: str) -> str | list[str]:
        prompt = render_prompt(tokenizer=tokenizer, question=question)
        results = []
        for format_name in format_names:
            try:
                results.append(generate_answer(pipe=pipelines[format_name], prompt=prompt, max_new_tokens=max_new_tokens)[0])
            except Exception as error:  # show the error of one format instead of stopping the demo
                results.append(f"Error: {error_line(error)}")
        # Gradio hands the whole return value to the only output when there is one, and one value to each output otherwise
        return results[0] if len(results) == 1 else results

    return gr.Interface(
        fn=answer_all,
        inputs=gr.Textbox(label="Question", lines=2),
        outputs=[gr.Textbox(label=format_name) for format_name in format_names],
        title="Compare LLM weight formats",
        flagging_mode="never",
    )


if pipelines:
    demo = make_demo(pipelines=pipelines, tokenizer=tokenizer, max_new_tokens=MAX_NEW_TOKENS)
    # if you are launching remotely, specify server_name and server_port
    # demo.launch(server_name='your server name', server_port='server port in int')
    # Read more in the docs: https://gradio.app/docs/
    demo.launch(debug=True)
else:
    print("No format is loaded: the demo needs at least one format that runs on the selected device.")

## Cleanup
[back to top ⬆️](#Table-of-contents:)

The notebook keeps the original checkpoint, the converted models and the compiled model cache next to it. Uncomment and run the cell below to delete them.

The cell does not delete the helper files that the first code cell fetched, the `__pycache__` folder next to them, the `.venv` folder with the installed packages, or the wikitext-2 dataset that the data-aware formats download into the Hugging Face cache.

In [ ]:
# # Uncomment and run this cell to delete the original checkpoint, the converted models and the compiled model cache.
# # Restart the kernel first, so that no loaded model keeps its files open.
# import os
# import shutil
# import stat
# import sys
# from collections.abc import Callable
# from pathlib import Path
#
#
# def remove_read_only(function: Callable[[str], object], path: str, _: object) -> None:
#     """
#     Clear the read-only flag of a file that could not be deleted (compiled model cache files have it on Windows) and
#     delete the file again.
#
#     :param function: The function that failed on the file.
#     :param path: Path of the file.
#     """
#     os.chmod(path, stat.S_IWRITE)
#     function(path)
#
#
# for folder in (Path("model"), Path("cache")):
#     if folder.exists():
#         if sys.version_info >= (3, 12):
#             shutil.rmtree(folder, onexc=remove_read_only)
#         else:
#             shutil.rmtree(folder, onerror=remove_read_only)